In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [2]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [4]:
from sklearn.ensemble import RandomForestRegressor

In [5]:
from sklearn.model_selection import train_test_split

In [6]:
from sklearn.metrics import (r2_score, mean_absolute_error)

---

In [7]:
# Step 1 : Create Dataset

data = pd.DataFrame({

    "Age":[
        22,25,28,30,35,
        40,45,50,55,60
    ],
    "AnnualIncome":[
        300000,
        400000,
        np.nan,
        550000,
        650000,
        750000,
        850000,
        950000,
        1100000,
        1250000
    ],
    "CreditScore":[
        600,
        620,
        650,
        680,
        np.nan,
        730,
        760,
        780,
        810,
        850
    ],
    "ExistingCreditLimit":[
        50000,
        70000,
        90000,
        120000,
        150000,
        180000,
        220000,
        260000,
        np.nan,
        350000
    ],
    "MonthlySpending":[
        10000,
        12000,
        15000,
        18000,
        22000,
        25000,
        28000,
        32000,
        36000,
        40000
    ],
    "NumberOfCreditCards":[
        1,1,2,2,3,
        3,4,4,5,5
    ],
    "NewCreditLimit":[
        60000,
        80000,
        100000,
        130000,
        170000,
        210000,
        250000,
        300000,
        350000,
        420000
    ]
})

In [8]:
print("\nOriginal Dataset:\n")
data


Original Dataset:



,Age,AnnualIncome,CreditScore,ExistingCreditLimit,MonthlySpending,NumberOfCreditCards,NewCreditLimit
0,22,300000.0,600.0,50000.0,10000,1,60000
1,25,400000.0,620.0,70000.0,12000,1,80000
2,28,NaN,650.0,90000.0,15000,2,100000
3,30,550000.0,680.0,120000.0,18000,2,130000
4,35,650000.0,NaN,150000.0,22000,3,170000
5,40,750000.0,730.0,180000.0,25000,3,210000
6,45,850000.0,760.0,220000.0,28000,4,250000
7,50,950000.0,780.0,260000.0,32000,4,300000
8,55,1100000.0,810.0,NaN,36000,5,350000
9,60,1250000.0,850.0,350000.0,40000,5,420000


---

In [9]:
# Step 2 : Features and Target

X = data.drop("NewCreditLimit", axis=1)

y = data["NewCreditLimit"]

---

In [10]:
# Step 3 : Handle Missing Values

imputer = SimpleImputer(strategy="mean")

X_imputed = imputer.fit_transform(X)

---

In [11]:
# Step 4 : Standardize Data

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_imputed)

---

In [12]:
# Step 5 : K-Means Clustering

kmeans = KMeans(n_clusters=4, random_state=42)

clusters = kmeans.fit_predict(X_scaled)

In [13]:
# Add cluster to dataset

data["CustomerCluster"] = clusters

print("\nCustomer Clusters:\n")
print(data[["AnnualIncome", "CreditScore", "CustomerCluster"]])


Customer Clusters:

   AnnualIncome  CreditScore  CustomerCluster
0      300000.0        600.0                2
1      400000.0        620.0                2
2           NaN        650.0                0
3      550000.0        680.0                0
4      650000.0          NaN                0
5      750000.0        730.0                0
6      850000.0        760.0                3
7      950000.0        780.0                3
8     1100000.0        810.0                1
9     1250000.0        850.0                3


---

In [14]:
# Step 6 : Add Cluster as Feature

X_final = pd.DataFrame(X_scaled, columns=["Age", "AnnualIncome", "CreditScore", "ExistingCreditLimit", "MonthlySpending", "NumberOfCreditCards"])

X_final["CustomerCluster"] = clusters

---

In [15]:
# Step 7 : Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

---

In [16]:
# Step 8 : Train Random Forest Regressor

model = RandomForestRegressor(n_estimators=100, random_state=42)

model = model.fit(X_train, y_train)

---

In [17]:
# Step 9 : Prediction

y_pred = model.predict(X_test)

---

In [18]:
# Step 10 : Evaluation

print(f"\nR2 Score : {r2_score(y_test, y_pred)}")

print(f"\nMean Absolute Error : {mean_absolute_error(y_test, y_pred)}")


R2 Score : 0.9559272976680384

Mean Absolute Error : 21850.0


---

In [19]:
# Step 11 : Feature Importance

importance = pd.DataFrame({"Feature": X_final.columns, "Importance": model.feature_importances_})

importance = importance.sort_values(by="Importance", ascending=False)

print("\nFeature Importance:\n")

print(importance)


Feature Importance:

               Feature  Importance
0                  Age    0.194134
2          CreditScore    0.171632
5  NumberOfCreditCards    0.158647
1         AnnualIncome    0.157127
3  ExistingCreditLimit    0.129633
4      MonthlySpending    0.118895
6      CustomerCluster    0.069931


---

In [20]:
# Step 12 : Predict New Customer

new_customer = pd.DataFrame({

    "Age":[38],
    "AnnualIncome":[700000],
    "CreditScore":[720],
    "ExistingCreditLimit":[170000],
    "MonthlySpending":[24000],
    "NumberOfCreditCards":[3]
})

In [21]:
# Handle missing values

new_customer = imputer.transform(new_customer)